In [24]:
%load_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
import mne
from pydeconv.utils import *
from pydeconv.pydeconv_sims_v2 import *
from pydeconv import *
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.metrics import mean_squared_error
%matplotlib qt 


# raw     = mne.io.read_raw_eeglab(data_path + "629959_analysis.set", preload=True)

# # Initialize the model

# rERP_model = PyDeconv(settings = settings , features = features, eeg = raw)
# X_design = rERP_model.create_matrix()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
# --- Build a compound ERP kernel ---
kernel = CompoundKernel(sfreq=500)
kernel.add(peak_latency=0.1,  amplitude=0.10, width=0.03, shape='gaussian', label='P1')
kernel.add(peak_latency=0.17, amplitude=-0.07, width=0.04, shape='gaussian', label='N170')
kernel.add(peak_latency=0.25, amplitude=0.04, width=0.05, shape='hanning',  label='P2')
kernel.plot()

# --- Create random event onsets ---
sfreq = 500
duration = 6000  # seconds
n_events = 100
latencies = np.sort(np.random.choice(np.arange(500, int(duration * sfreq) - 500), n_events, replace=False))
events = pd.DataFrame({'latency': latencies, 'type': 'stimulus'})

# --- Simulate ---
sim = EEGSimulator(sfreq=sfreq, duration=duration)
sim.add_kernel('stimulus', kernel)
sim.set_events(events)
sim.simulate()
sim.add_noise(colour='brown', scale=0.3)
sim.plot()

[0. 0. 0. ... 0. 0. 0.]
[-1.51582450e-18  1.12283297e-19 -1.25383014e-18 ... -1.78178348e-18
  6.08869676e-19  1.17061816e-18]
[0. 0. 0. ... 0. 0. 0.]


In [32]:
# Create sample TrialStructure
# Create a sample TrialStructure using the TrialVariable class

# Define trial variable
trial_result = TrialVariable(
    name='correct',
    generator=lambda n, _rng : np.random.choice([0, 1], size=n),
    static_accross_trial=True
)
trial_mss = TrialVariable(
    name='MSS',
    generator=lambda n, _rng : np.random.choice([1, 2, 3, 4], size=n),
    static_accross_trial=True
)
stim_type = TrialVariable(
    name='stimulus_type',
    generator=lambda n, rng: rng.choice(['fixation', 'saccade'], size=n)
)
rt_var = TrialVariable(
    name='sacc_amplitude',
    generator=lambda n, rng: rng.uniform(low=0.0, high=1.0, size=n)
)

# Specify trial structure configuration
trial_vars = [trial_result, trial_mss, stim_type, rt_var]

# Initialize TrialStructure with a number of trials (e.g., 100)
trial_struct = TrialStructure(sfreq=sfreq, variables=trial_vars)

for _ in range(5):
    # Generate the events DataFrame
    events_df = trial_struct.generate_events_df(n_samples=sfreq * duration, n_events=n_events)

    # Display the resulting DataFrame
    print(events_df.head())

   latency  correct  MSS stimulus_type  sacc_amplitude
0     2642        0    3       saccade        0.491550
1    10562        0    3      fixation        0.274811
2    15030        0    3      fixation        0.864077
3    19999        0    3      fixation        0.071743
4    21149        0    3      fixation        0.322900
   latency  correct  MSS stimulus_type  sacc_amplitude
0    10974        1    2      fixation        0.717070
1    34575        1    2      fixation        0.754950
2    48152        1    2      fixation        0.291042
3    61724        1    2       saccade        0.421257
4    65945        1    2       saccade        0.541041
   latency  correct  MSS stimulus_type  sacc_amplitude
0     5733        0    3      fixation        0.819685
1    64477        0    3       saccade        0.258978
2    93312        0    3      fixation        0.491957
3   116574        0    3      fixation        0.974579
4   135999        0    3       saccade        0.759788
   latency